# Testing BEIR NDCG Scoring

This notebook tests the BEIR NDCG computation, especially for cases with multiple relevant documents.

In [1]:
from beir.retrieval.evaluation import EvaluateRetrieval
import numpy as np

## Basic NDCG Formula

NDCG@k = DCG@k / IDCG@k

Where:
- DCG@k = sum_{i=1}^{k} (2^{rel_i} - 1) / log2(i + 1)
- IDCG@k = DCG@k for the ideal ranking (all relevant docs at top)

For binary relevance (rel = 0 or 1):
- Relevant doc contributes: (2^1 - 1) / log2(rank + 1) = 1 / log2(rank + 1)
- Non-relevant doc contributes: 0

In [2]:
def manual_dcg(qrels, results, k=None):
    """
    Compute DCG manually for verification.
    
    Args:
        qrels: dict of {doc_id: relevance} - ground truth relevance scores
        results: dict of {doc_id: score} - predicted scores (higher = better rank)
        k: cutoff for DCG computation (default: all results)
    
    Returns:
        DCG score
    """
    # Sort documents by score (descending) to get ranking
    ranked_docs = sorted(results.keys(), key=lambda d: results[d], reverse=True)
    
    if k is None:
        k = len(ranked_docs)
    
    dcg = 0.0
    for i, doc_id in enumerate(ranked_docs[:k]):
        rel = qrels.get(doc_id, 0)  # 0 if doc not in qrels
        # Position is 1-indexed in the formula
        dcg += (2**rel - 1) / np.log2(i + 2)  # i+2 because i is 0-indexed
    return dcg

def manual_idcg(qrels, k=None):
    """
    Compute ideal DCG (IDCG) - DCG with perfect ranking.
    
    Args:
        qrels: dict of {doc_id: relevance} - ground truth relevance scores
        k: cutoff for IDCG computation
    
    Returns:
        IDCG score
    """
    # Sort relevances in descending order (ideal ranking)
    sorted_rels = sorted(qrels.values(), reverse=True)
    
    if k is None:
        k = len(sorted_rels)
    
    idcg = 0.0
    for i, rel in enumerate(sorted_rels[:k]):
        idcg += (2**rel - 1) / np.log2(i + 2)
    return idcg

def manual_ndcg(qrels, results, k=None):
    """
    Compute NDCG manually for verification.
    
    Args:
        qrels: dict of {doc_id: relevance} - ground truth relevance scores
        results: dict of {doc_id: score} - predicted scores (higher = better rank)
        k: cutoff for NDCG computation (default: all results)
    
    Returns:
        NDCG score
    """
    dcg = manual_dcg(qrels, results, k)
    idcg = manual_idcg(qrels, k)
    if idcg == 0:
        return 0.0
    return dcg / idcg

## Test Case 1: Single Relevant Document

In [3]:
# Single query with 1 relevant document among 5
qrels_1 = {
    'q1': {'doc1': 1}  # Only doc1 is relevant
}

# Case 1a: Relevant doc at rank 1 (best case)
results_1a = {
    'q1': {'doc1': 1.0, 'doc2': 0.9, 'doc3': 0.8, 'doc4': 0.7, 'doc5': 0.6}
}

evaluator = EvaluateRetrieval()
ndcg_1a, _map, recall, precision = evaluator.evaluate(qrels_1, results_1a, [1, 5, 10])
print("Case 1a: Relevant doc at rank 1")
print(f"  BEIR NDCG: {ndcg_1a}")
print(f"  Manual NDCG@5: {manual_ndcg(qrels_1['q1'], results_1a['q1'], k=5):.4f}")

Case 1a: Relevant doc at rank 1
  BEIR NDCG: {'NDCG@1': 1.0, 'NDCG@5': 1.0, 'NDCG@10': 1.0}
  Manual NDCG@5: 1.0000


In [4]:
# Case 1b: Relevant doc at rank 2
results_1b = {
    'q1': {'doc2': 1.0, 'doc1': 0.9, 'doc3': 0.8, 'doc4': 0.7, 'doc5': 0.6}
}

ndcg_1b, _map, recall, precision = evaluator.evaluate(qrels_1, results_1b, [1, 5, 10])
print("Case 1b: Relevant doc at rank 2")
print(f"  BEIR NDCG: {ndcg_1b}")
print(f"  Manual NDCG@5: {manual_ndcg(qrels_1['q1'], results_1b['q1'], k=5):.4f}")

Case 1b: Relevant doc at rank 2
  BEIR NDCG: {'NDCG@1': 0.0, 'NDCG@5': 0.63093, 'NDCG@10': 0.63093}
  Manual NDCG@5: 0.6309


In [5]:
# Case 1c: Relevant doc at rank 5
results_1c = {
    'q1': {'doc2': 1.0, 'doc3': 0.9, 'doc4': 0.8, 'doc5': 0.7, 'doc1': 0.6}
}

ndcg_1c, _map, recall, precision = evaluator.evaluate(qrels_1, results_1c, [1, 5, 10])
print("Case 1c: Relevant doc at rank 5")
print(f"  BEIR NDCG: {ndcg_1c}")
print(f"  Manual NDCG@5: {manual_ndcg(qrels_1['q1'], results_1c['q1'], k=5):.4f}")

Case 1c: Relevant doc at rank 5
  BEIR NDCG: {'NDCG@1': 0.0, 'NDCG@5': 0.38685, 'NDCG@10': 0.38685}
  Manual NDCG@5: 0.3869


## Test Case 2: Two Relevant Documents (Both with Score 1)

In [7]:
# Two relevant documents
qrels_2 = {
    'q1': {'doc2': 1, 'doc1': 1}  # Both doc1 and doc2 are relevant
}

# Case 2a: Both relevant docs at top (ranks 1 and 2) - best case
results_2a = {
    'q1': {'doc1': 1.0, 'doc2': 0.9, 'doc3': 0.8, 'doc4': 0.7, 'doc5': 0.6}
}

ndcg_2a, _map, recall, precision = evaluator.evaluate(qrels_2, results_2a, [1, 5, 10])
print("Case 2a: Both relevant docs at ranks 1 and 2 (best case)")
print(f"  BEIR NDCG: {ndcg_2a}")
print(f"  Manual NDCG@5: {manual_ndcg(qrels_2['q1'], results_2a['q1'], k=5):.4f}")
print(f"  Recall: {recall}")

Case 2a: Both relevant docs at ranks 1 and 2 (best case)
  BEIR NDCG: {'NDCG@1': 1.0, 'NDCG@5': 1.0, 'NDCG@10': 1.0}
  Manual NDCG@5: 1.0000
  Recall: {'Recall@1': 0.5, 'Recall@5': 1.0, 'Recall@10': 1.0}


In [8]:
# Case 2b: Relevant docs at ranks 1 and 3
results_2b = {
    'q1': {'doc1': 1.0, 'doc3': 0.9, 'doc2': 0.8, 'doc4': 0.7, 'doc5': 0.6}
}

ndcg_2b, _map, recall, precision = evaluator.evaluate(qrels_2, results_2b, [1, 5, 10])
print("Case 2b: Relevant docs at ranks 1 and 3")
print(f"  BEIR NDCG: {ndcg_2b}")
print(f"  Manual NDCG@5: {manual_ndcg(qrels_2['q1'], results_2b['q1'], k=5):.4f}")

Case 2b: Relevant docs at ranks 1 and 3
  BEIR NDCG: {'NDCG@1': 1.0, 'NDCG@5': 0.91972, 'NDCG@10': 0.91972}
  Manual NDCG@5: 0.9197


In [9]:
# Case 2c: Relevant docs at ranks 2 and 3
results_2c = {
    'q1': {'doc3': 1.0, 'doc1': 0.9, 'doc2': 0.8, 'doc4': 0.7, 'doc5': 0.6}
}

ndcg_2c, _map, recall, precision = evaluator.evaluate(qrels_2, results_2c, [1, 5, 10])
print("Case 2c: Relevant docs at ranks 2 and 3")
print(f"  BEIR NDCG: {ndcg_2c}")
print(f"  Manual NDCG@5: {manual_ndcg(qrels_2['q1'], results_2c['q1'], k=5):.4f}")

Case 2c: Relevant docs at ranks 2 and 3
  BEIR NDCG: {'NDCG@1': 0.0, 'NDCG@5': 0.69343, 'NDCG@10': 0.69343}
  Manual NDCG@5: 0.6934


In [10]:
# Case 2d: Relevant docs at ranks 1 and 5
results_2d = {
    'q1': {'doc1': 1.0, 'doc3': 0.9, 'doc4': 0.8, 'doc5': 0.7, 'doc2': 0.6}
}

ndcg_2d, _map, recall, precision = evaluator.evaluate(qrels_2, results_2d, [1, 5, 10])
print("Case 2d: Relevant docs at ranks 1 and 5")
print(f"  BEIR NDCG: {ndcg_2d}")
print(f"  Manual NDCG@5: {manual_ndcg(qrels_2['q1'], results_2d['q1'], k=5):.4f}")

Case 2d: Relevant docs at ranks 1 and 5
  BEIR NDCG: {'NDCG@1': 1.0, 'NDCG@5': 0.85034, 'NDCG@10': 0.85034}
  Manual NDCG@5: 0.8503


In [11]:
# Case 2e: Relevant docs at ranks 4 and 5 (worst case within top-5)
results_2e = {
    'q1': {'doc3': 1.0, 'doc4': 0.9, 'doc5': 0.8, 'doc1': 0.7, 'doc2': 0.6}
}

ndcg_2e, _map, recall, precision = evaluator.evaluate(qrels_2, results_2e, [1, 5, 10])
print("Case 2e: Relevant docs at ranks 4 and 5")
print(f"  BEIR NDCG: {ndcg_2e}")
print(f"  Manual NDCG@5: {manual_ndcg(qrels_2['q1'], results_2e['q1'], k=5):.4f}")

Case 2e: Relevant docs at ranks 4 and 5
  BEIR NDCG: {'NDCG@1': 0.0, 'NDCG@5': 0.50127, 'NDCG@10': 0.50127}
  Manual NDCG@5: 0.5013


## Test Case 3: Understanding IDCG with 2 Relevant Documents

When there are 2 relevant documents with score=1, the ideal DCG is:
- IDCG = 1/log2(2) + 1/log2(3) = 1.0 + 0.631 = 1.631

In [12]:
# Compute IDCG for 2 relevant documents
idcg_2_docs = 1/np.log2(2) + 1/np.log2(3)
print(f"IDCG for 2 relevant documents: {idcg_2_docs:.4f}")
print(f"  Position 1 contribution: {1/np.log2(2):.4f}")
print(f"  Position 2 contribution: {1/np.log2(3):.4f}")

IDCG for 2 relevant documents: 1.6309
  Position 1 contribution: 1.0000
  Position 2 contribution: 0.6309


In [13]:
# Verify DCG calculations for case 2a (ranks 1 and 2)
dcg_2a = 1/np.log2(2) + 1/np.log2(3)  # Both at optimal positions
print(f"Case 2a DCG: {dcg_2a:.4f}")
print(f"Case 2a NDCG: {dcg_2a/idcg_2_docs:.4f} (should be 1.0)")

# Case 2b (ranks 1 and 3)
dcg_2b = 1/np.log2(2) + 1/np.log2(4)  # Position 3 = index 2, so log2(2+2)=log2(4)
print(f"\nCase 2b DCG: {dcg_2b:.4f}")
print(f"Case 2b NDCG: {dcg_2b/idcg_2_docs:.4f}")

# Case 2c (ranks 2 and 3)
dcg_2c = 1/np.log2(3) + 1/np.log2(4)
print(f"\nCase 2c DCG: {dcg_2c:.4f}")
print(f"Case 2c NDCG: {dcg_2c/idcg_2_docs:.4f}")

Case 2a DCG: 1.6309
Case 2a NDCG: 1.0000 (should be 1.0)

Case 2b DCG: 1.5000
Case 2b NDCG: 0.9197

Case 2c DCG: 1.1309
Case 2c NDCG: 0.6934


## Test Case 4: NDCG@1 with Multiple Relevant Documents

Important: NDCG@1 only looks at the first position. With 2 relevant docs, getting one at rank 1 gives NDCG@1=1.0.

In [14]:
# NDCG@1 comparison
print("NDCG@1 with 2 relevant documents:")
print(f"  Case 2a (rel at 1,2): NDCG@1 = {ndcg_2a.get('NDCG@1', 'N/A')}")
print(f"  Case 2c (rel at 2,3): NDCG@1 = {ndcg_2c.get('NDCG@1', 'N/A')}")
print(f"  Case 2e (rel at 4,5): NDCG@1 = {ndcg_2e.get('NDCG@1', 'N/A')}")

NDCG@1 with 2 relevant documents:
  Case 2a (rel at 1,2): NDCG@1 = 1.0
  Case 2c (rel at 2,3): NDCG@1 = 0.0
  Case 2e (rel at 4,5): NDCG@1 = 0.0


## Test Case 5: Graded Relevance (Scores > 1)

In [15]:
# Graded relevance: one highly relevant (score=2), one somewhat relevant (score=1)
qrels_graded = {
    'q1': {'doc1': 2, 'doc2': 1}  # doc1 is more relevant than doc2
}

# Case 5a: High relevance doc first
results_5a = {
    'q1': {'doc1': 1.0, 'doc2': 0.9, 'doc3': 0.8, 'doc4': 0.7, 'doc5': 0.6}
}

ndcg_5a, _map, recall, precision = evaluator.evaluate(qrels_graded, results_5a, [1, 5, 10])
print("Case 5a: High relevance (2) at rank 1, lower (1) at rank 2")
print(f"  BEIR NDCG: {ndcg_5a}")

# For graded relevance:
# DCG = (2^2-1)/log2(2) + (2^1-1)/log2(3) = 3/1 + 1/0.631 = 3 + 0.631 = 3.631
# IDCG is the same (optimal ordering)
print(f"  Manual calc: DCG = (2^2-1)/log2(2) + (2^1-1)/log2(3) = {(2**2-1)/np.log2(2) + (2**1-1)/np.log2(3):.4f}")

Case 5a: High relevance (2) at rank 1, lower (1) at rank 2
  BEIR NDCG: {'NDCG@1': 1.0, 'NDCG@5': 1.0, 'NDCG@10': 1.0}
  Manual calc: DCG = (2^2-1)/log2(2) + (2^1-1)/log2(3) = 3.6309


In [16]:
# Case 5b: Low relevance doc first (suboptimal)
results_5b = {
    'q1': {'doc2': 1.0, 'doc1': 0.9, 'doc3': 0.8, 'doc4': 0.7, 'doc5': 0.6}
}

ndcg_5b, _map, recall, precision = evaluator.evaluate(qrels_graded, results_5b, [1, 5, 10])
print("Case 5b: Low relevance (1) at rank 1, higher (2) at rank 2")
print(f"  BEIR NDCG: {ndcg_5b}")

# DCG = (2^1-1)/log2(2) + (2^2-1)/log2(3) = 1/1 + 3/1.585 = 1 + 1.893 = 2.893
# IDCG = 3 + 0.631 = 3.631
# NDCG = 2.893 / 3.631 = 0.797
dcg_5b = (2**1-1)/np.log2(2) + (2**2-1)/np.log2(3)
idcg_5 = (2**2-1)/np.log2(2) + (2**1-1)/np.log2(3)
print(f"  Manual calc: DCG = {dcg_5b:.4f}, IDCG = {idcg_5:.4f}, NDCG = {dcg_5b/idcg_5:.4f}")

Case 5b: Low relevance (1) at rank 1, higher (2) at rank 2
  BEIR NDCG: {'NDCG@1': 0.5, 'NDCG@5': 0.85972, 'NDCG@10': 0.85972}
  Manual calc: DCG = 2.8928, IDCG = 3.6309, NDCG = 0.7967


## Test Case 6: Multiple Queries

In [17]:
# Multiple queries with different numbers of relevant docs
qrels_multi = {
    'q1': {'doc1': 1, 'doc2': 1},  # 2 relevant docs
    'q2': {'doc3': 1},              # 1 relevant doc
    'q3': {'doc4': 1, 'doc5': 1, 'doc6': 1}  # 3 relevant docs
}

results_multi = {
    'q1': {'doc1': 1.0, 'doc2': 0.9, 'doc7': 0.8},  # Both relevant at top
    'q2': {'doc7': 1.0, 'doc3': 0.9, 'doc8': 0.8},  # Relevant at rank 2
    'q3': {'doc4': 1.0, 'doc7': 0.9, 'doc5': 0.8, 'doc6': 0.7}  # 1st, 3rd, 4th
}

ndcg_multi, _map, recall, precision = evaluator.evaluate(qrels_multi, results_multi, [1, 5, 10])
print("Multiple queries with different relevant doc counts:")
print(f"  BEIR NDCG: {ndcg_multi}")
print(f"  Recall: {recall}")

Multiple queries with different relevant doc counts:
  BEIR NDCG: {'NDCG@1': 0.66667, 'NDCG@5': 0.84565, 'NDCG@10': 0.84565}
  Recall: {'Recall@1': 0.27778, 'Recall@5': 1.0, 'Recall@10': 1.0}


In [18]:
# Verify per-query NDCG
print("\nPer-query verification:")

# Q1: rel at ranks 1, 2 -> NDCG@5 = 1.0
q1_qrels = {'doc1': 1, 'doc2': 1}
q1_results = {'doc1': 1.0, 'doc2': 0.9, 'doc7': 0.8}
print(f"  Q1 (rel at 1,2): NDCG = {manual_ndcg(q1_qrels, q1_results, k=5):.4f}")

# Q2: rel at rank 2 -> NDCG@5 = 1/log2(3) / 1/log2(2) = 0.631
q2_qrels = {'doc3': 1}
q2_results = {'doc7': 1.0, 'doc3': 0.9, 'doc8': 0.8}
print(f"  Q2 (rel at 2): NDCG = {manual_ndcg(q2_qrels, q2_results, k=5):.4f}")

# Q3: rel at ranks 1, 3, 4
q3_qrels = {'doc4': 1, 'doc5': 1, 'doc6': 1}
q3_results = {'doc4': 1.0, 'doc7': 0.9, 'doc5': 0.8, 'doc6': 0.7}
print(f"  Q3 (rel at 1,3,4): NDCG = {manual_ndcg(q3_qrels, q3_results, k=5):.4f}")

# Average
avg = (manual_ndcg(q1_qrels, q1_results, k=5) + 
       manual_ndcg(q2_qrels, q2_results, k=5) + 
       manual_ndcg(q3_qrels, q3_results, k=5)) / 3
print(f"\n  Average NDCG@5: {avg:.4f}")


Per-query verification:
  Q1 (rel at 1,2): NDCG = 1.0000
  Q2 (rel at 2): NDCG = 0.6309
  Q3 (rel at 1,3,4): NDCG = 0.9060

  Average NDCG@5: 0.8457


## Test Case 7: Missing Documents in Results

What happens when relevant documents are not in the results?

In [19]:
# Qrels has 2 relevant docs, but results only include 1
qrels_missing = {
    'q1': {'doc1': 1, 'doc2': 1}  # Both relevant
}

# Results don't include doc2
results_missing = {
    'q1': {'doc1': 1.0, 'doc3': 0.9, 'doc4': 0.8}
}

ndcg_missing, _map, recall, precision = evaluator.evaluate(qrels_missing, results_missing, [1, 5, 10])
print("Missing relevant document in results:")
print(f"  BEIR NDCG: {ndcg_missing}")
print(f"  Recall: {recall}")
print("\nNote: IDCG is still computed based on all relevant docs in qrels,")
print("so missing a relevant doc in results will reduce NDCG.")

Missing relevant document in results:
  BEIR NDCG: {'NDCG@1': 1.0, 'NDCG@5': 0.61315, 'NDCG@10': 0.61315}
  Recall: {'Recall@1': 0.5, 'Recall@5': 0.5, 'Recall@10': 0.5}

Note: IDCG is still computed based on all relevant docs in qrels,
so missing a relevant doc in results will reduce NDCG.


In [20]:
# Manual verification
# DCG = 1/log2(2) = 1.0 (only doc1 at rank 1)
# IDCG = 1/log2(2) + 1/log2(3) = 1.631 (both docs at optimal positions)
# NDCG = 1.0 / 1.631 = 0.613
dcg_missing = 1/np.log2(2)
idcg_missing = 1/np.log2(2) + 1/np.log2(3)
print(f"Manual calculation:")
print(f"  DCG = {dcg_missing:.4f}")
print(f"  IDCG = {idcg_missing:.4f}")
print(f"  NDCG = {dcg_missing/idcg_missing:.4f}")

Manual calculation:
  DCG = 1.0000
  IDCG = 1.6309
  NDCG = 0.6131


## Summary Table: NDCG Values for 2 Relevant Documents

In [21]:
import pandas as pd

# Create summary table for 2 relevant documents at different rank positions
# qrels always has doc1 and doc2 as relevant
qrels_summary = {'doc1': 1, 'doc2': 1}

# Helper to create results dict with doc1 at rank r1 and doc2 at rank r2
def make_results(r1, r2):
    """Create results dict with doc1 at rank r1, doc2 at rank r2 (1-indexed)."""
    docs = ['doc1', 'doc2', 'doc3', 'doc4', 'doc5']
    scores = {}
    used = set()
    # Place doc1 at rank r1
    scores['doc1'] = 1.0 - (r1 - 1) * 0.1
    used.add(r1)
    # Place doc2 at rank r2
    scores['doc2'] = 1.0 - (r2 - 1) * 0.1
    used.add(r2)
    # Fill remaining positions with non-relevant docs
    non_rel_idx = 0
    for rank in range(1, 6):
        if rank not in used:
            while f'doc{non_rel_idx + 3}' in scores:
                non_rel_idx += 1
            scores[f'doc{non_rel_idx + 3}'] = 1.0 - (rank - 1) * 0.1
            non_rel_idx += 1
    return scores

scenarios = [
    ("Ranks 1, 2", 1, 2),
    ("Ranks 1, 3", 1, 3),
    ("Ranks 1, 4", 1, 4),
    ("Ranks 1, 5", 1, 5),
    ("Ranks 2, 3", 2, 3),
    ("Ranks 2, 4", 2, 4),
    ("Ranks 2, 5", 2, 5),
    ("Ranks 3, 4", 3, 4),
    ("Ranks 3, 5", 3, 5),
    ("Ranks 4, 5", 4, 5),
]

data = []
for name, r1, r2 in scenarios:
    results = make_results(r1, r2)
    data.append({
        'Scenario': name,
        'NDCG@1': f"{manual_ndcg(qrels_summary, results, k=1):.4f}",
        'NDCG@5': f"{manual_ndcg(qrels_summary, results, k=5):.4f}",
        'NDCG@10': f"{manual_ndcg(qrels_summary, results, k=10):.4f}",
    })

df = pd.DataFrame(data)
print("NDCG values for different placements of 2 relevant documents:")
print(df.to_string(index=False))

NDCG values for different placements of 2 relevant documents:
  Scenario NDCG@1 NDCG@5 NDCG@10
Ranks 1, 2 1.0000 1.0000  1.0000
Ranks 1, 3 1.0000 0.9197  0.9197
Ranks 1, 4 1.0000 0.8772  0.8772
Ranks 1, 5 1.0000 0.8503  0.8503
Ranks 2, 3 0.0000 0.6934  0.6934
Ranks 2, 4 0.0000 0.6509  0.6509
Ranks 2, 5 0.0000 0.6241  0.6241
Ranks 3, 4 0.0000 0.5706  0.5706
Ranks 3, 5 0.0000 0.5438  0.5438
Ranks 4, 5 0.0000 0.5013  0.5013


## Key Takeaways

1. **IDCG depends on total relevant docs**: With 2 relevant docs, IDCG = 1/log2(2) + 1/log2(3) = 1.631

2. **Position matters logarithmically**: Moving a doc from rank 1 to rank 2 loses more NDCG than moving from rank 4 to rank 5

3. **NDCG@1 is binary for binary relevance**: Either you have a relevant doc at rank 1 (NDCG@1=1) or you don't (NDCG@1=0)

4. **Missing docs in results hurt NDCG**: If a relevant doc is not retrieved, it contributes 0 to DCG but still inflates IDCG

5. **For 2 relevant docs with binary relevance**:
   - Best case (ranks 1,2): NDCG@5 = 1.0
   - Worst case (ranks 4,5): NDCG@5 ≈ 0.56